# Feed data generation

### Input data

In [3]:
# All crop data
SPAM_BASE_DIR = "../data/raw/SPAM_2010"

# Above ground biomass data
BIOMASS_RASTER = "../data/raw/Global_Maps_C_Density_2010_1763/data/aboveground_biomass_carbon_2010.tif"

# Administrative boundary data
GAUL_BOUNDARIES_ADM0 = "../data/raw/GAUL/gaul0_asap/gaul0_asap.shp"
GAUL_BOUNDARIES_ADM1 = "../data/raw/GAUL/gaul1_asap/gaul1_asap.shp"
# https://data.europa.eu/euodp/data/dataset/jrc-10112-10004
GADM_BOUNDARIES_CHN_ADM0 = "../data/raw/GADM/gadm36_CHN_shp/gadm36_CHN_0.shp"
GADM_BOUNDARIES_CHN_ADM1 = "../data/raw/GADM/gadm36_CHN_shp/gadm36_CHN_1.shp"

### Method - crops

**(A) Prepare statistical boundaries**
1. Create a geo-dataframe of administrative boundaries. ADM0, ADM1, ADM2 are the three levels. GADM data shall be used for ADM1 and ADM2 in China, GAUL for everywhere else, matching what was done in Qiangyi et al.

**(B) Prepare reference tables**
1. Load in a pre-generated table of crop biophysical data
2. Load in a pre-generated table of crop intensities (harvest frequencies over a year), organised by area, crop type and system (e.g. irrigated)

**(C) Calculate residues**
1. Iterate through SPAM_BASE_DIR (contains files for each crop)
2. Zonal stats production data to (A)
3. Calculate new production values for each pixel based on (B2)
4. Convert production to residues and nutritional data using (B1)

### (A) Prepare statistical boundaries

In [4]:
import geopandas as gpd
import pandas as pd

## ADM0
boundaries_adm0 = gpd.read_file(GAUL_BOUNDARIES_ADM0)
# Trim and re-name GAUL ADM0 columns
boundaries_adm0 = boundaries_adm0[["asap0_id", "name0", "km2_tot", "isocode", "geometry"]]
boundaries_adm0.columns = ["adm0_id", "adm0_name", "area_km2", "isocode", "geometry"]
boundaries_adm0.adm0_name = boundaries_adm0.adm0_name.apply(str.lower)
boundaries_adm0 = boundaries_adm0.set_index("adm0_name")

## ADM1
# Trim and re-name GAUL ADM1 columns
gaul_adm1 = gpd.read_file(GAUL_BOUNDARIES_ADM1)
gaul_adm1 = gaul_adm1[["asap1_id", "name1", "name0", "asap0_id", "km2_tot", "geometry"]]
gaul_adm1.columns = ["adm1_id", "adm1_name", "adm0_name", "adm0_id", "area_km2", "geometry"]
gaul_adm1.set_index("adm1_id")

# Trim and re-name GADM ADM1 columns
gadm_chn_adm1 = gpd.read_file(GADM_BOUNDARIES_CHN_ADM1)
gadm_chn_adm1 = gadm_chn_adm1[["GID_0", "NAME_0", "GID_1", "NAME_1", "geometry"]]
gadm_chn_adm1.columns = ["adm0_id", "adm0_name", "adm1_id", "adm1_name", "geometry"]
gadm_chn_adm1.set_index("adm1_id")

# Combine
boundaries_adm1 = gaul_adm1[gaul_adm1.adm0_name != "China"].append(gadm_chn_adm1)
boundaries_adm1.adm1_name = boundaries_adm1.adm1_name.apply(str.lower)
boundaries_adm1 = boundaries_adm1.set_index("adm1_name")

In [71]:
# Process crop intensity
crop_intensity = pd.read_csv("../data/external/cropping_intensity_all.csv")

In [72]:
# check that all name_admins are in the adm1 or adm0 file

def check(na):
    na = na.lower()
    return na in boundaries_adm0.index or na in boundaries_adm1.index

assert sum(crop_intensity.name_admin.apply(check)) == len(crop_intensity), f"Need more boundaries data"

AssertionError: Need more boundaries data

In [73]:
# Create one dataframe for each crop production system, this allows us to create a 1-1-1 relationship between a dataframe, 
# set of administrative boundaries and corresponding raster file.

crop_intensity["name_admin"] = crop_intensity["name_admin"].apply(str.strip)
crop_intensity["name_cntr"] = crop_intensity["name_cntr"].apply(str.strip)

crop_intensity["name_admin_lower"] = crop_intensity.name_admin.apply(str.lower)
crop_intensity["name_cntr_lower"] = crop_intensity.name_cntr.apply(str.lower)

crop_intensity = crop_intensity[crop_intensity.name_admin_lower != "name unknown"]
crop_intensity = crop_intensity[crop_intensity.name_admin_lower != "administrative unit not available"]
crop_intensity["name_full_lower"] = crop_intensity.apply(lambda r: f"{r.name_cntr.lower()}-{r.name_admin.lower()}", axis=1)

crop_intensity_irrigated = crop_intensity[crop_intensity.rec_type == 'CIIRR']
crop_intensity_rainfed_high_inputs = crop_intensity[crop_intensity.rec_type == 'CIRFH']
crop_intensity_rainfed_low_inputs = crop_intensity[crop_intensity.rec_type == 'CIRFL']
crop_intensity_rainfed_subsistence = crop_intensity[crop_intensity.rec_type == 'CIRFL']

In [76]:
crop_intensity_irrigated[crop_intensity_irrigated.name_admin == "Mexico"]

,iso3,prod_level,name_cntr,name_admin,rec_type,unit,wheat,rice,maize,barley,...,plantain,trop_fruit,temp_fruit,vegetable,rest_crop,year_data,source,name_admin_lower,name_cntr_lower,name_full_lower
1795,MEX,MX00,MEXICO,Mexico,CIIRR,nr,1.03,1.0,1.11,1.15,...,0.0,1.08,1.06,1.04,1.01,NaN,NaN,mexico,mexico,mexico-mexico
1840,MEX,MX15,MEXICO,Mexico,CIIRR,nr,1.00,1.0,1.00,1.00,...,1.0,1.00,1.00,1.00,1.00,NaN,NaN,mexico,mexico,mexico-mexico


In [69]:
# Ensure we have a unique set of values for each administrative boundary

for df in [crop_intensity_irrigated, crop_intensity_rainfed_high_inputs, 
           crop_intensity_rainfed_low_inputs, crop_intensity_rainfed_subsistence]:
    assert len(df) == len(df.name_full_lower.unique()), f"{df.rec_type.unique()} is not unique"

AssertionError: ['CIIRR'] is not unique

In [57]:
# crop_intensity_irrigated[crop_intensity_irrigated.name_admin_lower.duplicated()]
crop_intensity.adm_full_lower

0        afghanistan-afghanistan
1        afghanistan-afghanistan
2        afghanistan-afghanistan
3          bangladesh-bangladesh
4          bangladesh-bangladesh
                  ...           
3752    united states- wisconsin
3753    united states- wisconsin
3754      united states- wyoming
3755      united states- wyoming
3756      united states- wyoming
Name: adm_full_lower, Length: 3736, dtype: object

In [27]:
import pandas as pd

a = pd.merge(boundaries_adm0, crop_intensity, right_on="name_admin_lower", left_index=True)

### (B) Prepare crop biophysical data

In [82]:
import pandas as pd
import os

biophysical_lookup = pd.read_csv("../data/processed/residue_lookup.csv", index_col="crop")
crops_in_lookup = list(biophysical_lookup.spam_name)
spam_schema = pd.read_csv("../data/processed/spam_2010_crop_schema.csv")
crops_in_schema = list(sp)

In [102]:
print([(f["name"], f["spam_name"]) for i, f in spam_schema.iterrows() if f.spam_name not in crops_in_lookup])

[('potato', 'pota'), ('yams', 'yams'), ('other roots', 'orts'), ('chickpea', 'chic'), ('cowpea', 'cowp'), ('pigeonpea', 'pige'), ('lentil', 'lent'), ('coconut', 'cnut'), ('tropical fruit', 'trof'), ('temperate fruit', 'temf'), ('vegetables', 'vege'), ('sesameseed', 'sesa'), ('other oil crops', 'ooil'), ('other fibre crops', 'ofib'), ('arabica coffee', 'acof'), ('robusta coffee', 'rcof'), ('cocoa', 'coco'), ('tea', 'teas'), ('tobacco', 'toba'), ('rest of crops', 'rest')]


### (C) Calculate residues

Calculate residues
want these files:

- *_TI	irrigated portion of crop
- *_TH	rainfed high inputs portion of crop
- *_TL	rainfed low inputs portion of crop
- *_TS	rainfed subsistence portion of crop


From Ulrike:

"The column rec_type identifies the production system for which the next columns are valid:

CIIRR: cropping intensity for irrigated crops

CIRFH: cropping intensity for rainfed high-input crops

CIRFL: cropping intensity for rainfed low-input crops

For subsistence production systems we take CIRFL."

In [1]:
import os

production_tifs = [files for _, _, files in os.walk("../data/raw/SPAM_2010/spam2010v2r0_global_prod.geotiff")][0]

In [ ]:
# We'll try first rasterizing 

image = features.rasterize(
            ((g, 255) for g, v in shapes),
            out_shape=src.shape,
            transform=src.transform)